In [ ]:
from utils.util_octree_stuff import *
from utils.util_sample_stuff import *
from utils.util_visual_stuff import *
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from dataset.voxel_dataset import VoxelPatchDataset, get_dataloader

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [10]:

DEPTH_STOP = 6
LATENT_DIM = 10
FULL_DEPTH = 3
PATCH_DIR = "data/patches_128x128x64_s16"
loader = get_dataloader(PATCH_DIR, batch_size=1)
from models.networks.dualoctree_networks.graph_sem_vae import GraphVAE

model = GraphVAE(
    depth=6,
    channel_in=32,     
    nout=2,
    full_depth=FULL_DEPTH,
    depth_stop=DEPTH_STOP ,
    depth_out=6,
    latent_dim=LATENT_DIM,  
    num_classes=92,# latent dimension
    resblk_num=1,
    
).cuda()

# model.load_state_dict(torch.load("vae6_8_5.pt"),model.parameters())

model.train()

GraphVAE(
  (patch_embed): SharedPatchEmbed(
    (embedding): Embedding(92, 32)
    (conv_proj): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (3): ReLU()
      (4): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (5): ReLU()
    )
    (to_latent): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=256, out_features=32, bias=True)
    )
  )
  (conv1): GraphConv(channel_in=32, channel_out=32, n_edge_type=7, avg_degree=7, n_node_type=5)
  (encoder): ModuleList(
    (0): GraphResBlocks(
      (resblks): ModuleList()
    )
  )
  (downsample): ModuleList()
  (encoder_norm_out): DualOctreeGroupNorm(in_channels=32, group=8, nempty=False)
  (nonlinearity): GELU(approximate='none')
  (decoder): ModuleList(
    (0): GraphResBlocks(
      (resblks): ModuleList(
        (0): GraphResBlock(
     

In [15]:
import tqdm
from models.networks.dualoctree_networks.graph_sem_vae import GraphVAE

T_max = 100

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max, eta_min=1e-7)
for epoch in range(100):
    pbar = tqdm.tqdm(loader, desc=f"Epoch {epoch}")
    model.train()
    
    for vox, _ in pbar:
        vox = vox.cuda().long()
        
        # 1. Create Octree directly from voxel occupancy
        # (Assuming your points2octree logic just uses non-empty voxel locations)
        # If possible, move this logic into your Dataset class to speed up training!
        non_empty_mask = get_non_empty_mask(vox[0], 2)
        points = voxel_grid_to_points(non_empty_mask)
        octree = points2octree(points, depth=6, full_depth=FULL_DEPTH).cuda()
        
        # 2. Forward Pass
        # Pass features directly if your model supports it
        patch_feat = voxel_to_patch(vox, patch_size=2)
        assign_octree_patch_features(patch_feat[0], octree, 6)
        
        output = model(octree, octree_out=octree)
        
        # 3. Compute Losses
        # Use the 'clean' semantic loss we just built
        sem_loss_dict = compute_semantic_loss_clean(
            output['sem_voxs'], 
            output['octree_out'], 
            vox, 
            torch.ones(92).cuda()
        )
        
        oct_loss_dict = compute_octree_loss(output['logits'], octree)
        
  
        
        # Combine all losses efficiently
        total_loss = output['kl_loss'] + sem_loss_dict["sem_loss_6"]
        total_loss += sum(v for k, v in oct_loss_dict.items() if 'loss_' in k)

        # 4. Optimization
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

    
        pbar.set_postfix({
            "loss": f"{total_loss.item():.3f}",
            "sem": f"{sem_loss_dict['sem_loss_6'].item():.3f}",
            "acc": f"{sem_loss_dict['sem_accu_6'].item():.3f}"
        })
        break
    break

    scheduler.step()
    torch.save(model.state_dict(), f"saved_model/vae_model_{epoch}.pt")

Epoch 0:   0%|          | 0/281 [00:00<?, ?it/s]/tmp/ipykernel_18033/892053679.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(self.files[idx], map_lo

In [16]:
recon_voxel = reconstruct_voxel_from_patch(
    sem_voxs=output['sem_voxs'],
    octree=output['octree_out'],
    depth=6,
    shape=(1, 128, 128, 64)
)


In [17]:
visualize_voxel_labels(recon_voxel[0], voxel_size=0.05, origin=(0,0,0))

Visualizing voxel grid: shape=(128, 128, 64), filled=62113
